# 1. Anchor/Rare 및 Utility Router 학습

이 노트북은 LLM을 호출하지 않습니다. 먼저 ARVO semantic feature로 Anchor + Rare Trigger Router를 학습합니다. 별도로 수집한 Expert×Model outcome JSONL이 있으면 실제 성공 확률을 예측하는 Utility Router도 학습합니다. 기존 Softmax Router 코드는 비교 baseline으로만 남아 있습니다.

In [1]:
import json
import sys
from collections import Counter
from pathlib import Path
from pprint import pprint

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

DATA_DIR = ROOT / 'data' / 'phase2e'
ARTIFACT_DIR = ROOT / 'artifacts' / 'phase2e'
ANCHOR_MODEL_PATH = ARTIFACT_DIR / 'router_anchor_rare_v2.pkl'
UTILITY_MODEL_PATH = ARTIFACT_DIR / 'router_top2_full5_v4.pkl'
SUMMARY_PATH = ARTIFACT_DIR / 'router_training_summary.json'
SEED = 2026
TARGET_RARE_RECALL = 0.95
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

from llm_security.datasets import load_router_samples_jsonl, load_utility_samples_jsonl
from llm_security.models import to_dict
from llm_security.routing import (
    AnchorRareRouter,
    BudgetedUtilityRouter,
    UtilityPolicyConfig,
    assert_project_disjoint,
    split_gate_calibration_samples,
)

## 데이터 확인

phase2e-prepare가 만든 project-disjoint semantic Router JSONL을 사용합니다. 다중 label을 제거하지 않습니다.

In [2]:
train_path = DATA_DIR / 'semantic' / 'router_train.jsonl'
dev_path = DATA_DIR / 'semantic' / 'router_dev.jsonl'
if not train_path.exists() or not dev_path.exists():
    raise FileNotFoundError(
        '먼저 python -m llm_security.cli phase2e-prepare --backend semantic 명령을 실행하세요.'
    )
train_samples = load_router_samples_jsonl(train_path)
dev_samples = load_router_samples_jsonl(dev_path)
schemas = {row.candidate.feature_schema_version for row in [*train_samples, *dev_samples]}
if schemas != {'semantic-cwe-v2'}:
    raise RuntimeError(
        f'Expected semantic-cwe-v2, got {sorted(schemas)}. Rerun phase2e-prepare.'
    )
print('train/dev:', len(train_samples), len(dev_samples))
print('train labels:', Counter(label.value for row in train_samples for label in row.labels))

train/dev: 2563 413
train labels: Counter({'memory_bounds': 1446, 'control_state_error': 884, 'lifetime_resource': 233})


## Anchor + Rare Trigger 학습

memory_bounds와 control_state_error는 항상 실행합니다. 나머지 family는 독립 binary trigger로 학습하며 dev에서 rare recall 95%를 우선해 threshold를 정합니다.

In [3]:
anchor_router = AnchorRareRouter.fit(train_samples, seed=SEED)
anchor_calibration = anchor_router.calibrate_threshold(
    dev_samples, target_rare_recall=TARGET_RARE_RECALL
)
anchor_dev_metrics = anchor_router.evaluate(dev_samples)
anchor_router.save(ANCHOR_MODEL_PATH)
anchor_summary = {
    'artifact': str(ANCHOR_MODEL_PATH),
    'anchors': [item.value for item in anchor_router.anchors],
    'rare_families': [item.value for item in anchor_router.rare_families],
    'calibration': to_dict(anchor_calibration),
    'dev_metrics': to_dict(anchor_dev_metrics),
}
pprint(anchor_summary)

{'anchors': ['memory_bounds', 'control_state_error'],
 'artifact': 'C:\\Users\\junhyun111\\Desktop\\llm-security\\artifacts\\phase2e\\router_anchor_rare_v2.pkl',
 'calibration': {'achieved_recall': 0.9508196721311475,
                 'rare_trigger_rate': 0.8837772397094431,
                 'target_met': True,
                 'target_recall': 0.95,
                 'threshold': 0.27875041158774877},
 'dev_metrics': {'average_experts_per_candidate': 2.883777239709443,
                 'exact_coverage': 0.9927360774818402,
                 'expert_coverage': 0.9927360774818402,
                 'llm_calls_saved_vs_all_six': 1287,
                 'rare_precision': 0.1589041095890411,
                 'rare_recall': 0.9508196721311475,
                 'rare_trigger_rate': 0.8837772397094431,
                 'sample_count': 413},
 'rare_families': ['lifetime_resource']}


## Adaptive Top-2 / Full-5 Utility Router 학습

5개 Expert의 전체 outcome matrix로 성공확률을 학습합니다. dev 프로젝트는 Escalation Gate 학습과 threshold calibration으로 다시 분리하며 test outcome은 이 노트북에서 읽지 않습니다.

In [4]:
utility_dir = ROOT / 'data' / 'utility'
utility_train_path = utility_dir / 'outcomes_train.jsonl'
utility_dev_path = utility_dir / 'outcomes_dev.jsonl'
utility_summary = {'trained': False, 'reason': 'outcome JSONL not found'}
if utility_train_path.exists() and utility_dev_path.exists():
    utility_train = load_utility_samples_jsonl(utility_train_path)
    utility_dev = load_utility_samples_jsonl(utility_dev_path)
    assert_project_disjoint(
        utility_train, utility_dev, first_name='train', second_name='dev'
    )
    gate_rows, calibration_rows = split_gate_calibration_samples(
        utility_dev, seed=SEED, gate_fraction=0.5
    )
    utility_router = BudgetedUtilityRouter.fit(
        utility_train,
        policy=UtilityPolicyConfig(escalation_threshold=0.85),
        seed=SEED,
    )
    gate_candidates = utility_router.fit_escalation_gate(gate_rows, seed=SEED)
    escalation_calibration = utility_router.calibrate_threshold(
        calibration_rows, target_truth_recall=0.95
    )
    baseline_calibration = utility_router.calibrate_baselines(calibration_rows)
    utility_metrics = utility_router.evaluate(calibration_rows)
    utility_router.save(UTILITY_MODEL_PATH)
    utility_summary = {
        'trained': True,
        'artifact': str(UTILITY_MODEL_PATH),
        'train_rows': len(utility_train),
        'dev_rows': len(utility_dev),
        'gate_training_candidates': gate_candidates,
        'calibration': to_dict(escalation_calibration),
        'baseline_calibration': to_dict(baseline_calibration),
        'calibration_metrics': to_dict(utility_metrics),
    }
pprint(utility_summary)

{'reason': 'outcome JSONL not found', 'trained': False}


In [5]:
summary = {
    'seed': SEED,
    'llm_api_calls_during_training': 0,
    'anchor_rare': anchor_summary,
    'utility': utility_summary,
}
SUMMARY_PATH.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2) + '\n', encoding='utf-8'
)
AnchorRareRouter.load(ANCHOR_MODEL_PATH)
if utility_summary['trained']:
    BudgetedUtilityRouter.load(UTILITY_MODEL_PATH)
print('artifact validation: PASS')
print('summary:', SUMMARY_PATH)

artifact validation: PASS
summary: C:\Users\junhyun111\Desktop\llm-security\artifacts\phase2e\router_training_summary.json
